# INTRODUCTION
CNN pour faire des prédictions. 

In [1]:
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.io import read_image
from sklearn.metrics import accuracy_score, recall_score
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm
import itertools
import random

# ✅ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using", device)

Using cpu


In [2]:
# ✅ Custom Dataset (filtre les classes 0 et 1)
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, limit=10000):
        self.samples = []
        self.transform = transform
        for patient in os.listdir(root_dir):
            patient_dir = os.path.join(root_dir, patient)
            if not os.path.isdir(patient_dir):
                continue
            for label in ['0', '1']:
                label_dir = os.path.join(patient_dir, label)
                if os.path.isdir(label_dir):
                    for fname in os.listdir(label_dir):
                        img_path = os.path.join(label_dir, fname)
                        self.samples.append((img_path, int(label)))

        # Mélange et limite
        random.shuffle(self.samples)
        self.samples = self.samples[:limit]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = read_image(path).float() / 255.0  # normalisation simple
        if self.transform:
            image = self.transform(image)
        return image, label


In [3]:
# ✅ Transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
])

# ✅ Chargement du dataset
root_dir = "../data/image50_clahe"  # mets le bon chemin ici
dataset = CustomImageDataset(root_dir, transform=transform)
print("Total images loaded:", len(dataset))

# ✅ Split train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

Total images loaded: 10000


In [4]:
# ✅ CNN Model dynamique
class SimpleCNN(nn.Module):
    def __init__(self, conv_layers, fc_units):
        super(SimpleCNN, self).__init__()
        layers = []
        in_channels = 3
        for _ in range(conv_layers):
            layers.append(nn.Conv2d(in_channels, 16, kernel_size=3, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2))
            in_channels = 16
        self.conv = nn.Sequential(*layers)

        self.fc1 = nn.Linear(16 * (64 // (2 ** conv_layers))**2, fc_units)
        self.fc2 = nn.Linear(fc_units, 2)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
# ✅ Param grid
param_grid = {
    "conv_layers": [1, 2, 3],
    "fc_units": [64, 128, 256],
    "lr": [0.01, 0.001, 0.0001],
}

best_acc = 0
best_model = None

# ✅ Grid search
for conv_layers, fc_units, lr in itertools.product(*param_grid.values()):
    print(f"🔍 Testing conv_layers={conv_layers}, fc_units={fc_units}, lr={lr}")
    model = SimpleCNN(conv_layers, fc_units).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    loop = tqdm(train_loader, desc="Training", leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())

    # Eval
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    print(f"✅ conv_layers={conv_layers}, fc_units={fc_units}, lr={lr} — Acc={acc:.4f}, Recall={recall:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_model = model

print("🎯 Best val accuracy:", best_acc)

🔍 Testing conv_layers=1, fc_units=64, lr=0.01


Training:   0%|          | 0/250 [00:00<?, ?it/s]

✅ conv_layers=1, fc_units=64, lr=0.01 — Acc=0.7165, Recall=0.0000
🔍 Testing conv_layers=1, fc_units=64, lr=0.001


Training:   0%|          | 0/250 [00:00<?, ?it/s]

✅ conv_layers=1, fc_units=64, lr=0.001 — Acc=0.8060, Recall=0.5371
🔍 Testing conv_layers=1, fc_units=64, lr=0.0001


...

Training:   0%|          | 0/250 [00:00<?, ?it/s]

✅ conv_layers=1, fc_units=256, lr=0.0001 — Acc=0.7930, Recall=0.4470
🔍 Testing conv_layers=2, fc_units=64, lr=0.01


Training:   0%|          | 0/250 [00:00<?, ?it/s]

✅ conv_layers=2, fc_units=64, lr=0.01 — Acc=0.7170, Recall=0.0000
🔍 Testing conv_layers=2, fc_units=64, lr=0.001


Training:   0%|          | 0/250 [00:00<?, ?it/s]